# Unit 11 exercises: crashes, families of cars, and surviving them

A crash restarts the race, so a crash costs nearly everything driven so far, and a car that crashes in every attempt never finishes.
These exercises measure that cost, choose a fraction by the expected race time over a family of cars, explore racekit's perturbed family, build a respawn detector and the warning instruments, and put a fallback into Unit 10's agent and race it on a car its MPC cannot drive.
Then they re-tune the fraction on a small training family and check it on a held-out car.
The Challenges try two things the lesson did not: braking on a sideslip warning, and a fraction for the unseen track.
Every number here is a number about racekit's **carla-like car** and its perturbed family: a model, not the truth, and the family's ranges are the book's model of model error, not a measurement of CARLA.

Run the setup cell first.
Each exercise names the variables your answer must define; fill in every `None` and `"TODO"`.
Functions and races from one exercise are reused in later ones, so run the exercises in order.
Every race uses `carla_like(noise=True, ...)` through the setup's `race()` helper (race seed k mod 5 for car k unless an exercise says otherwise) and `MPC_BUDGET_MS = None` (`NO_TIMING`).
An MPC race takes several seconds, a pure-pursuit race about half that; the exercises cut races short (`max_seconds`) wherever the question allows, and keep their number small.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import PERTURBATION_FAMILY, carla_like, perturbation_factors
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track
from scipy.optimize import lsq_linear

AGENTS = Path("../../projects/racekit/agents")
UNIT_09, UNIT_10, UNIT_11 = AGENTS / "unit-09.py", AGENTS / "unit-10.py", AGENTS / "unit-11.py"
workdir = Path(mkdtemp(prefix="roar-unit-11-ex-"))
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the error from the race's own line
TICK = 0.05 + 1e-9  # s: one control tick (race times are multiples of it)
DT = 0.05  # s: one control tick
HOLD_S = 1.0  # s: the respawn hold (20 ticks)
NO_TIMING = {"MPC_BUDGET_MS": None}  # every MPC race here: no wall clock inside the race
TRAINING, HELD_OUT = tuple(range(24)), tuple(range(24, 48))  # the perturbed family's two halves
QUIET, CRASH = (0.0, 0.0, 0.0), (0.0, -1000.0, 0.0)  # N s: collision readings, nothing and racekit's crash impulse
MEASURED_MU_G_2026 = (7.70, 7.69, 7.70, 9.81, 8.16, 8.04)  # Unit 8's grip probes on test-circuit-seed2026 (m/s^2)


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, written into workdir.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def race(source, k=None, track="test-circuit", seed=None, lag_ticks=None, max_seconds=450.0):
    # One race on car k of the perturbed family (k = None: the unperturbed car), with its noise.
    # The race seed is k mod 5 unless one is given (0 for the unperturbed car).
    if seed is None:
        seed = 0 if k is None else k % 5
    car = carla_like(noise=True, perturb=k, lag_ticks=lag_ticks)
    return run_race(source, track, car=car, seed=seed, max_seconds=max_seconds)


# Unit 9's line solver: a cubic B-spline of offsets, the smoothest line by bounded least squares.
def spline_basis(n, step=10):
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def normals(x, y):
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def second_difference(values):
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def solve_line(track, margin=2.5, pin=30):
    x, y = track.x, track.y
    nx, ny = normals(x, y)
    basis = spline_basis(len(x))
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    rows, columns = basis.shape
    index = np.arange(rows)
    pinned = (index < pin) | (index >= rows - pin)
    high = np.zeros(columns)
    for j in range(columns):
        support = basis[:, j] > 0
        high[j] = 0.0 if np.sum(pinned[support]) > 0 else np.min(track.half_width[support] - margin)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x


# Unit 9's line-relative clean rule; by default against test-circuit's line as the agents ship it.
with shim_installed():
    Unit11, _ = load_submission(UNIT_11)
line_offsets = np.array(Unit11(load_track("test-circuit").waypoints, None).offsets)


def line_error(result, offsets=None):
    offsets = line_offsets if offsets is None else offsets
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result, offsets=None):
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR

## Exercise 1 - The cost of a crash

The lesson's rule: a crash at time t costs t + 1 s, to within a second, because the restart is a fresh race from rest.
Check it on a car the lesson did not race.
`agents/unit-11.py` at Unit 10's 0.58 (`variant(UNIT_11, "u11-0.58", MU_FRACTION=0.58, **NO_TIMING)`) crashes once on held-out car 32, falls back, and finishes.
Race it on car 32 (`race(source, 32)`: race seed 2, `max_seconds=450.0`).

Its **fresh twin** is the same car and race seed, started in the fallback configuration: `variant(UNIT_11, "u11-twin", STEERING="pursuit", MU_FRACTION=0.58 * 0.86, **NO_TIMING)`.
Race the twin with `repeat_races` on `"test-circuit"`, `n=3, seeds=(2, 3, 4)`, `car_factory=lambda: carla_like(noise=True, perturb=32)` and `max_seconds=450.0`: its first race is the same-seed twin, and the three together show the noise.

Write:

- `crash_costs(result)`: the list of t + `HOLD_S` for every respawn time t of a race result (`result.respawn_times`).

Define:

- `crash_costs`: the function;
- `crash_race`: the race result, and `crash_time`: its first respawn time (s);
- `held_rows`: the number of telemetry rows with `held == 1`, and `hold_seconds`: `held_rows * DT`;
- `restart_progress`: the `progress` of the first telemetry row after the crash with `held == 0`;
- `remainder`: the race time after the restart, `crash_race.total_time - crash_time - hold_seconds`;
- `twin_results`: the `RepeatResult`, `twin_time`: its first race's time, and `twin_spread`: its spread (s);
- `twin_gap`: `remainder - twin_time`, and `within_a_second`: `True` when |`twin_gap`| ≤ 1.0;
- `systematic`: `True` when |`twin_gap`| is larger than `twin_spread` (a difference the noise does not explain);
- `fig`, `ax`: `progress` against `t` for the crashed race and the same-seed twin, with a legend, axis labels and a title;
- `why_shorter`: two sentences: the remainder is a little shorter than the fresh twin; what does racekit's runner reset at a respawn, what does it not, and which part of the agent carries over?

In [ ]:
def crash_costs(result):
    return None  # TODO: [t + HOLD_S for each respawn]


crash_race, crash_time = None, None  # TODO
held_rows, hold_seconds = None, None  # TODO
restart_progress = None  # TODO
remainder = None  # TODO
twin_results, twin_time, twin_spread = None, None, None  # TODO
twin_gap, within_a_second = None, None  # TODO
systematic = None  # TODO
fig, ax = None, None  # TODO
why_shorter = "TODO"

## Exercise 2 - Expected race time, and the best fraction

The book's tests (plan `roar-010`, premise U11.2 and U11.4) raced Unit 10's agent at four fractions on the 24 training cars of the family (race seed k mod 5, a 700 s limit).
The cell below gives, per fraction, how many races were clean and their mean time, and every race that was not clean (car, race time, finished).
A race counts at **min(its time, the cap)** if it finished and at the cap if it did not: with a lower cap, a slow finish becomes a DNF.

Write:

- `capped_time(time, finished, cap)`: one race's time as counted;
- `expected_time(n_clean, clean_mean, not_clean, cap)`: the mean over all the races of a fraction, the clean ones at `clean_mean` (their mean is all you need) and each not-clean one by `capped_time`.

Define:

- `capped_time`, `expected_time`: the functions;
- `expected_700`: a dict from each fraction to its expected race time with a 700 s cap, and `matches_premise`: `True` when every one is within 0.01 s of `PREMISE_MEAN`;
- `best_700`: the fraction with the shortest expected race time;
- `expected_450`, `best_450`: the same with a 450 s cap (assume every clean race is under 450 s);
- `step_gain`: how much faster a clean race is at 0.56 than at 0.54 (from `CLEAN`), and `crash_cost_056`: car 8's race time at 0.56 minus the clean mean at 0.56;
- `break_even`: `step_gain / crash_cost_056`, the crash rate per race below which 0.56 would beat 0.54, and `races_per_crash`: `1 / break_even`;
- `fig`, `ax`: the expected race time against the fraction for both caps, with a legend, axis labels and a title;
- `why_not_the_edge`: two sentences: why does a fraction one step below the highest clean one win on expected time, and why can 24 cars not tell you whether the faster fraction pays?

In [ ]:
CAP = 700.0  # s: the book's tests' cap
FAMILY_SIZE = 24
# plan roar-010, premise U11.2/U11.4: Unit 10 on training cars 0-23 (race seed k mod 5, 700 s cap).
CLEAN = {0.58: (21, 275.05), 0.56: (22, 277.43), 0.54: (24, 279.95), 0.52: (24, 282.58)}  # (count, mean time in s)
NOT_CLEAN = {  # (car, race time in s, finished)
    0.58: [(8, 471.85, True), (16, 700.0, False), (20, 700.0, False)],
    0.56: [(8, 474.00, True), (9, 486.90, True)],
    0.54: [],
    0.52: [],
}
PREMISE_MEAN = {0.58: 318.66, 0.56: 294.35, 0.54: 279.95, 0.52: 282.58}  # s: the premise's means (700 s cap)


def capped_time(time, finished, cap):
    return None  # TODO


def expected_time(n_clean, clean_mean, not_clean, cap):
    return None  # TODO


expected_700, matches_premise = {}, None  # TODO
best_700 = None  # TODO
expected_450, best_450 = {}, None  # TODO
step_gain, crash_cost_056 = None, None  # TODO
break_even, races_per_crash = None, None  # TODO
fig, ax = None, None  # TODO
why_not_the_edge = "TODO"

## Exercise 3 - The perturbed family

`perturbation_factors(k)` returns car k's six factors (a dict from the names in `PERTURBATION_FAMILY` to a number, 1.0 meaning the car Unit 8 measured); `PERTURBATION_FAMILY` maps each name to its (low, high) range.

Define:

- `factors`: a dict from k = 0 … 47 to `perturbation_factors(k)`;
- `inside_ranges`: `True` when every factor of every car lies inside its range;
- `same_again`: `True` when `perturbation_factors(5)` equals `factors[5]` (drawn again);
- `grip_order`: the training cars sorted by their grip factor, lowest first;
- `effective_fraction`: a dict from each training car to `0.58 / grip`: the share of **that car's** grip that a planner using 0.58 of the **measured** grip really asks for (Unit 8's measured grip is the unperturbed car's);
- `over_edge`: the training cars whose effective fraction is above 0.60 (MPC's family edge in Unit 10), in `grip_order`.

Then race Unit 10's agent as shipped (0.58, `variant(UNIT_10, "u10-0.58", **NO_TIMING)`) on the **two** lowest-grip training cars, `grip_order[:2]`, each with `race(source, k, max_seconds=300.0)` (a clean race takes about 277 s).

- `family_races`: a dict from those two cars to their race results, and `crashed`: a dict from each to `result.collisions > 0`;
- `crash_ranks`: a dict from each car of `PREMISE_CRASHED_058` (the cars that were not clean at 0.58 in the book's tests) to its position in `grip_order` (1 = the lowest grip);
- `fig`, `ax`: a scatter of grip factor (x) against throttle factor (y) for the training cars, with the cars of `PREMISE_CRASHED_058` drawn with a different marker, a legend, axis labels and a title;
- `grip_story`: two sentences: does the grip factor alone predict which cars crash at 0.58? Use your two races and the throttle factors.

In [ ]:
PREMISE_CRASHED_058 = (8, 16, 20)  # not clean at 0.58 in the book's tests (premise U11.2)

factors = {}  # TODO
inside_ranges = None  # TODO
same_again = None  # TODO
grip_order = None  # TODO
effective_fraction = {}  # TODO
over_edge = None  # TODO
family_races, crashed = {}, {}  # TODO
crash_ranks = {}  # TODO
fig, ax = None, None  # TODO
grip_story = "TODO"

## Exercise 4 - A respawn detector

Nobody tells the agent that it crashed; it has its collision sensor, `self.collision_sensor.get_last_gym_observation()`, a 3-vector impulse (N·s).
racekit reports the crash impulse on the first step after the hold, on that one step; CARLA's sensor may repeat its last reading.

Write the lesson's edge rule twice.

First as a function, `edge_rule(readings, threshold=100.0, refractory=40)`: given one reading per step (steps counted from 1), return the list of steps on which a respawn is handled: the reading's norm (`np.linalg.norm`) is over `threshold`, the reading is not equal to the previous step's, and no respawn was handled in the last `refractory` steps (a step s is allowed when s − the last handled step ≥ `refractory`).

Then inside an agent.
Load Unit 10 at 0.58 (`with shim_installed(): Unit10At058, _ = load_submission(variant(UNIT_10, "u10-0.58", **NO_TIMING))`) and write `class Detector(Unit10At058)` whose `__init__` calls `super().__init__(*args, **kwargs)`, sets up its own memory (a step counter, the previous reading, the step of the last respawn handled, and `self.handled`, a list of the steps it handled) and appends itself to a list `detectors`; and whose `async def step(self)` counts the step, applies the rule to this step's reading (as a tuple of floats), records a handled step, and then does everything Unit 10 does: `await super().step()`.
Race `Detector` on car 20 (`race(Detector, 20, max_seconds=300.0)`): Unit 10 crashes twice there in 300 s.

Define:

- `edge_rule`, `Detector`, `detectors`: the function, the class and the list;
- `script_handled`: `edge_rule(SCRIPT)`, and `script_level`: the number of readings in `SCRIPT` whose norm is over 100 (what a level-triggered detector would count);
- `detector_race`: the race result, and `detector`: the agent (`detectors[0]`);
- `count_matches`: `True` when `len(detector.handled)` equals `detector_race.collisions`;
- `detection_times`: the time of each handled step (the step-th driving row, `held == 0`, of the telemetry: one row per `step()` call), and `delays`: each detection time minus its respawn time (s);
- `why_edge`: one sentence: what would a level-triggered detector do on CARLA, and what does the 40-step window add to "not the previous reading"?

In [ ]:
SCRIPT = [QUIET] * 3 + [CRASH] * 8 + [QUIET] * 10 + [CRASH] + [QUIET] * 40 + [CRASH] * 2  # one reading per step


def edge_rule(readings, threshold=100.0, refractory=40):
    return None  # TODO: the steps handled


detectors = []
Detector = None  # TODO: the class
script_handled, script_level = None, None  # TODO
detector_race, detector = None, None  # TODO
count_matches = None  # TODO
detection_times, delays = None, None  # TODO
why_edge = "TODO"

## Exercise 5 - Sideslip and line error, logged

The sideslip β is the angle from the car's heading to its velocity: β = atan2(v_y, v_x) − ψ, wrapped to [−180°, 180°), with (v_x, v_y) the velocity sensor's reading and ψ the yaw (`self.rpy_sensor.get_last_gym_observation()[2]`).
`agents/unit-11.py` computes it every step as `self.beta` (degrees), and its line error is `self.cross_track_error` (m).

Write:

- `my_beta(velocity, yaw)`: β in degrees from a velocity reading and a yaw (rad), wrapped with `(angle + math.pi) % (2 * math.pi) - math.pi` before converting;
- `logger(source)`: returns `(Logged, log)`, where `Logged` is a subclass of the submission's class (loaded with `load_submission` inside `shim_installed()`) whose `step` first awaits `super().step()` and then appends to `log`, a dict of lists: `"beta"` (`self.beta`), `"mine"` (`my_beta` of this step's velocity and yaw readings), `"error"` (`self.cross_track_error`) and `"speed"` (the velocity's norm).

With `source = variant(UNIT_11, "u11-0.58", MU_FRACTION=0.58, **NO_TIMING)`, race a fresh `Logged` on car 16 (`max_seconds=195.0`: it crashes at about 190.6 s) and another on car 0 (`max_seconds=450.0`: a clean race).
Turn each log into arrays, and add `"t"`: the telemetry's `t` on the driving rows (`held == 0`), one per step.

Define:

- `my_beta`, `logger`: the functions;
- `crash_race`, `crash_log`, `clean_race`, `clean_log`: the two races and their logs (dicts of arrays);
- `beta_agrees`: `True` when `my_beta` and `self.beta` differ by less than 1e-9 deg on every step of both races;
- `peak_beta_clean`, `peak_error_clean`: the largest |β| (deg) and |line error| (m) of the clean race, over the steps above 5 m/s;
- `lead_beta`, `lead_error`: the **lead time** of |β| over 8° and of |line error| over 1.6 m before car 16's crash (its first respawn time): how long before the crash the **last** run of consecutive steps (above 5 m/s) over the threshold began, the run that a warning would be part of (s; `None` if never over it);
- `fig`, `(ax_beta, ax_error)`: β and the line error against the time to the crash over the last 10 s before it, each with its warning threshold (±) and the clean race's peak (±) as horizontal lines, axis labels and titles;
- `enough_time`: two sentences: at 30 m/s, how far does the car travel in `lead_beta`, and what could an agent still do in that distance?

In [ ]:
def my_beta(velocity, yaw):
    return None  # TODO: degrees


def logger(source):
    return None, None  # TODO: (Logged, log)


crash_race, crash_log = None, None  # TODO
clean_race, clean_log = None, None  # TODO
beta_agrees = None  # TODO
peak_beta_clean, peak_error_clean = None, None  # TODO
lead_beta, lead_error = None, None  # TODO
fig, (ax_beta, ax_error) = None, (None, None)  # TODO
enough_time = "TODO"

## Exercise 6 - A warning threshold table

A warning is useful when it comes **early** before a crash and **never** when nothing is wrong.
Use Exercise 5's two logs (no new races).

For a threshold, the **lead time** is how long before car 16's crash the last run over it began (as in Exercise 5: a crossing early in the race, followed by a return below the threshold, is not a warning of this crash), and the **false alarms** are the steps of the clean race (above 5 m/s) on which it was over it.
A run of consecutive false-alarm steps is one **episode**.

Define:

- `beta_table`: a dict from each β threshold in `BETA_THRESHOLDS` to `(lead time or None, false-alarm steps, false-alarm episodes)`;
- `error_table`: the same for the line error and `ERROR_THRESHOLDS`;
- `quiet_beta`, `quiet_error`: the lowest threshold of each instrument with **no** false alarm in the clean race, and `quiet_leads`: a dict `{"beta": ..., "error": ...}` of their lead times;
- `fig`, `ax`: lead time (y) against false-alarm steps (x), one marker per threshold, the two instruments in different colours with a legend, axis labels and a title (leave out thresholds with no lead time);
- `best_trigger`: two or three sentences: which instrument and threshold would you use, and for what: to count, or to act? Why can one crash and one clean race not settle it (the lesson's warning trigger on the 24 training cars)?

In [ ]:
BETA_THRESHOLDS = (4.0, 5.0, 6.0, 8.0, 10.0, 12.0)  # deg
ERROR_THRESHOLDS = (1.0, 1.2, 1.4, 1.6, 1.8)  # m

beta_table, error_table = {}, {}  # TODO
quiet_beta, quiet_error = None, None  # TODO
quiet_leads = {}  # TODO
fig, ax = None, None  # TODO
best_trigger = "TODO"

## Exercise 7 - A fallback for Unit 10, on the lag-2 car

The lag-2 car, `carla_like(noise=True, lag_ticks=2)` (`race(source, lag_ticks=2)`), has **the lag Unit 8 identified, plus one tick**, and is otherwise the unperturbed car.
Unit 10's MPC crash-loops on it; this exercise gives Unit 10's agent its own fallback, and races it there.

Load Unit 10 at 0.54: `with shim_installed(): Unit10At054, _ = load_submission(variant(UNIT_10, "u10-0.54", MU_FRACTION=0.54, **NO_TIMING))`.
The agent's constants you need: `TARGET_SPEED = 45.0` m/s, `A_DEC = 5.0` and `A_ACC = 4.0` m/s² (Unit 7's profile), `MPC_LAG = 1` (Unit 10), and pure pursuit's lookahead `min(max(PP_K * v, PP_MIN), PP_MAX)` with `PP_K = 0.5` s, `PP_MIN = 3.0` m and `PP_MAX = 40.0` m.

Write:

- `corner_speed(kappa, a_lat)`: √(a_lat / |κ|), but at most `TARGET_SPEED` (and `TARGET_SPEED` when |κ| ≤ 1e-9);
- `plan_profile(cap, segment_length, a_dec, a_acc)`: Unit 7's forward–backward profile: starting from the caps, the backward pass v_i ≤ √(v_{i+1}² + 2 a_dec ds_i) walking the lap backwards, then the forward pass v_{i+1} ≤ √(v_i² + 2 a_acc ds_i), each pass **twice** around the lap (it is a loop);
- `class MyFallback(Unit10At054)` with:
  - `__init__`: set `self.scale = 1.0` **before** calling `super().__init__(*args, **kwargs)` (the parent's constructor plans its profile with `lateral_limit`), then Exercise 4's memory, `self.respawns = 0`, `self.reanchor = False`, `self.mode = "primary"`, and append itself to a list `fallbacks`;
  - `lateral_limit(self, section)`: `self.scale` times the parent's (`super().lateral_limit(section)`);
  - `replan(self)`: `self.cap` from `corner_speed(self.curvature[i], self.lateral_limit(self.section[i]))` for every waypoint i, and `self.profile = plan_profile(self.cap, self.segment_length, 5.0, 4.0)`;
  - `fall_back(self)`: clear the MPC's memory (`self.mpc_queue = [0.0] * 1`, `self.mpc_last = 0.0`, `self.plan = None`), set `self.reanchor = True` and `self.mode = "fallback"`, multiply `self.scale` by 0.86, and `replan()`;
  - `mpc_command(self, yaw, speed)`: in fallback mode, the agent's own pure pursuit from the location sensor's reading, `self.pursuit(x, y, yaw, speed)`; otherwise the parent's `mpc_command`;
  - `async def step(self)`: count the step and apply Exercise 4's edge rule to the collision reading; on a respawn, count it and `fall_back()`; otherwise, if `self.reanchor` is set, re-anchor (`self.current_index = self.nearest_index(location)`, the location sensor's reading) and clear it; then `await super().step()`.

Define:

- `corner_speed`, `plan_profile`, `MyFallback`, `fallbacks`: the functions, the class and the list;
- `profile_matches`: `True` when a fresh `MyFallback(load_track("test-circuit").waypoints, None)` has the same profile (`np.allclose`, `rtol=1e-12`) before and after calling its `replan()` (your profile is Unit 7's);
- `lookaheads`: a dict from speed (5.0, 15.0, 30.0, 45.0 m/s) to the pursuit lookahead (m);
- `plain_lag2`: Unit 10 at 0.54 on the lag-2 car with `max_seconds=100.0`, and `plain_respawns`: its respawn times;
- `my_lag2`: a fresh `MyFallback` on the lag-2 car (`max_seconds=450.0`), and `my_agent`: its agent (the **last** entry of `fallbacks`: `profile_matches` builds one too); `survives`: `True` when it finished with exactly one collision and `my_agent.mode == "fallback"`;
- `known_lag`: Unit 10 at 0.54 with `MPC_LAG=2` on the lag-2 car (`max_seconds=450.0`), and `known_lag_clean`: `clean(known_lag)`;
- `fallback_explained`: two or three sentences: why does the fallback change the controller instead of only slowing down, and why is `MPC_LAG = 2` not a fix you could ship to CARLA?

In [ ]:
def corner_speed(kappa, a_lat):
    return None  # TODO


def plan_profile(cap, segment_length, a_dec, a_acc):
    return None  # TODO: a list of speeds


fallbacks = []
MyFallback = None  # TODO: the class
profile_matches = None  # TODO
lookaheads = {}  # TODO
plain_lag2, plain_respawns = None, None  # TODO
my_lag2, my_agent, survives = None, None, None  # TODO
known_lag, known_lag_clean = None, None  # TODO
fallback_explained = "TODO"

## Exercise 8 - A robust fraction, by the margin rule

Re-tune the fraction the way Units 7 to 10 did, on a small training family, and check it on a held-out car.

- **The training family:** the three lowest-grip training cars, `grip_order[:3]` (Exercise 3).
- **The edge:** for f = 0.58, then 0.56, then 0.54, race Unit 10 at f (`variant(UNIT_10, ..., MU_FRACTION=f, **NO_TIMING)`) on the family's cars in grip order, with race seed **0** for every car (`seed=0`, as the probe record did) and `max_seconds=300.0`.
  Stop racing a fraction at its first car that is not clean (the fraction is then not clean on the family), and stop the search at the first fraction that is clean on all three: that is the **edge**.
- **The margin rule:** ship one grid step (0.02) inside the edge.
- **The noise:** the book's tests race car k with race seed k mod 5, not 0.
  Race the edge fraction once more on car 8 (a family car that is clean at the edge with seed 0) with its own race seed, 3 (`race(source, 8, max_seconds=300.0)`).
- **The held-out check:** race the shipped fraction on the lowest-grip **held-out** car (from `factors`), race seed 0, `max_seconds=450.0`.

Define:

- `small_family`: the three cars;
- `edge_races`: a dict from (f, k) to the result, for every race of the search, and `family_clean`: a dict from each fraction searched to `True` or `False`;
- `edge`: the edge fraction, and `shipped`: the margin rule's fraction (`round(edge - 0.02, 2)`);
- `noise_race`: the race of car 8 at the edge with race seed 3, and `edge_moves`: `True` when it is not clean;
- `held_out_car`, `held_out_race`, `held_out_clean`: the held-out car, its race, and `clean(held_out_race)`;
- `races_run`: how many races this exercise raced;
- `is_it_enough`: two or three sentences: a 3-car family picked by grip, one seed per car: what could this procedure miss, and what did the lesson's 24 + 24 cars and the seed2026 finding add?

In [ ]:
small_family = None  # TODO
edge_races, family_clean = {}, {}  # TODO
edge, shipped = None, None  # TODO
noise_race, edge_moves = None, None  # TODO
held_out_car, held_out_race, held_out_clean = None, None, None  # TODO
races_run = None  # TODO
is_it_enough = "TODO"

## Exercise 9 - Challenge: brake on the warning

The lesson acted on the sideslip warning by **falling back** (pure pursuit, a lower speed plan), and it did not save a single car.
Try a different reaction: **brake**.
While the agent is in its primary mode, above 5 m/s, and |β| > `BRAKE_BETA`, lower the speed goal to `speed - BRAKE_DROP`; otherwise change nothing.

The speed goal comes from the agent's `profile_goal(self, speed)` method, called after the warning is computed in each step (so `self.beta` is this step's).
Make a function `braking(source, brake_beta, brake_drop)` that returns a subclass of the submission's class whose `profile_goal` returns `min(super().profile_goal(speed), speed - brake_drop)` when the condition holds, and counts those steps in `self.brakings`.

Use `agents/unit-11.py` at 0.58 (`variant(UNIT_11, "u11-0.58", MU_FRACTION=0.58, **NO_TIMING)`) on car 20, whose crash at 0.58 comes at about 99 s on race seed 0.
Is a save on one seed a save?
Race car 20 on seeds 0, 1 and 2 with `repeat_races(..., "test-circuit", n=3, seeds=(0, 1, 2), car_factory=lambda: carla_like(noise=True, perturb=20), max_seconds=110.0)`, once with the plain file and once with `braking(source, 8.0, 3.0)`.

Define:

- `braking`: the function;
- `plain_runs`, `braked_runs`: the two `RepeatResult` objects;
- `plain_crashed`, `braked_crashed`: dicts from seed to `True` when that race had a collision;
- `saved_seeds`: the seeds on which the plain file crashed and the braking one did not;
- `crash_times`: a dict from seed to `(plain crash time, braked crash time)` where both crashed;
- `verdict`: two or three sentences: did braking save car 20 on any seed? If it had saved one seed, why would that still not be evidence (Unit 1's rule, the lesson's 24 cars), and what would you race next?

In [ ]:
def braking(source, brake_beta, brake_drop):
    return None  # TODO: the class


plain_runs, braked_runs = None, None  # TODO
plain_crashed, braked_crashed = {}, {}  # TODO
saved_seeds = None  # TODO
crash_times = {}  # TODO
verdict = "TODO"

## Exercise 10 - Challenge: a fraction for `test-circuit-seed2026`

The lesson's finding: the 0.54 that is clean on all 48 perturbed cars on `test-circuit` crashes on 4 of 12 on `test-circuit-seed2026` (premise U11.8: cars 6, 8, 9 and 10, race seed 0).
Find a fraction that is clean there.

Solve `test-circuit-seed2026`'s line (`seed2026 = load_track("test-circuit-seed2026")`, `solve_line(seed2026)`) and write it, with `MEASURED_MU_G_2026`, into `agents/unit-11.py` (`variant(UNIT_11, ..., LINE_CONTROL=tuple(float(v) for v in control), MU_G=MEASURED_MU_G_2026, MU_FRACTION=f, **NO_TIMING)`).
For f = 0.52, then 0.50, then 0.48, race cars 6 and 9 (`race(source, k, track="test-circuit-seed2026", seed=0, max_seconds=300.0)`), stopping at the first fraction clean on both.

Define:

- `seed2026_control`, `seed2026_offsets`: the line's control values and its offsets (`spline_basis(len(seed2026)) @ seed2026_control`);
- `seed2026_races`: a dict from (f, k) to the result, `seed2026_clean`: a dict from (f, k) to `clean(result, seed2026_offsets)`;
- `seed2026_fraction`: the first fraction clean on both cars;
- `slower_by`: the mean race time of your two clean races at that fraction minus 282.55 s (the premise's median over cars 0-11 at 0.54 there);
- `one_fraction`: two or three sentences: Checkpoint B forbids tuning per track. Would you ship your fraction everywhere, given what one step costs on `test-circuit` (Exercise 2), and what would you race before deciding?

In [ ]:
seed2026 = load_track("test-circuit-seed2026")
seed2026_control, seed2026_offsets = None, None  # TODO
seed2026_races, seed2026_clean = {}, {}  # TODO
seed2026_fraction = None  # TODO
slower_by = None  # TODO
one_fraction = "TODO"